# P6｜BEATs Token-Level Temporal Refinement

**状态：Deferred / Scientific HOLD；非首轮必要条件。** 目的：在 P2 的四数据集 shared-window package 之后，仅把 HF 的 2-s window-level temporal route细化为 BEATs token-level tokens/mask/time-map route。

## 唯一变量、匹配对照与四数据集边界

matched comparator=P2。P2 已让四数据集共享 BEATs window encoder与projector；P6只作为后续 temporal refinement，把 HF window embeddings 改为 BEATs token sequence。由于时间粒度、projector application与head同时变化，P6−P2只能解释为 temporal package-level effect。必须先逐值通过 ICBHI/SPRSound/KAUH window route parity。

- ICBHI cycle flat4、SPRSound event binary/raw7、KAUH recording raw9 继续完全复用 P2 shared-window route；KAUH B/D/E 同 patient group。
- HF 以共享 token 时间区间对齐四个 native temporal channels：I、E、CAS、DAS；每通道独立保留 observation_mask 与 valid_mask。raw conservative policy 中 missing/unknown/gap 全部 masked；P6 执行候选冻结为 source-paper-native one-vs-rest rasterization，interval 外的零仅是 source-task constructed negative，不是 raw negative、shared normal 或 P8 shared evidence。
- 输入为 16 kHz source-time-lineaged windows；统一 encoder 输出为 tokens float [B,L,D]、token_mask bool [B,L]、time_map float [B,L,2]（source seconds, half-open）、pooled float [B,D]；HF temporal logits 固定为 [B,L,4]，对应 I/E/CAS/DAS。

In [ ]:
from pathlib import Path
import os
from baseline.multidataset_pipeline.preflight import HF_NATIVE_METRICS, P6TokenTemporalHead, freeze_receipt, hf_masked_channel_balanced_bce

TEMPORAL_CONTRACT = {
    "tokens": ["B", "L", "D"],
    "token_mask": ["B", "L"],
    "time_map": ["B", "L", 2],
    "pooled": ["B", "D"],
    "projected_tokens": ["B", "L", 256],
    "hf_channels": ["I", "E", "CAS", "DAS"],
    "hf_logits": ["B", "L", 4],
    "hf_observation_mask": ["B", "L", 4],
    "hf_valid_mask": ["B", "L", 4],
    "time_unit": "source_seconds_half_open",
}
PIPELINE = {
    "id": "P6",
    "comparator": "P2_window_level_temporal_package",
    "first_round_required": False,
    "seed": 20260728,
    "split_policy": "reuse_P2_immutable_receipts",
    "encoder": "BEATs_shared_temporal_adapter",
    "hf_head": "shared_projector_output_plus_biased_linear_256_to_4",
    "hf_loss": "equal_channel_mean_of_masked_BCEWithLogits",
    "hf_metrics": HF_NATIVE_METRICS,
    "hf_target_policy": "PAPER_NATIVE_RASTERIZED_OVR",
    "hf_alignment": "token_center_in_interval",
    "hf_negative_semantics": "source_task_constructed_not_raw_normal",
    "hf_shared_label_eligible": False,
    "hf_source_policy_reference": "docs/datasets/four_dataset_task_contract_review_2026-07-28.md",
    "non_hf_gate": "exact_shared_window_route_parity_with_P2",
    "contract_modules": ["baseline.multidataset_pipeline.contracts", "baseline.multidataset_pipeline.beats_temporal", "baseline.multidataset_pipeline.hf_data", "baseline.multidataset_pipeline.preflight"],
    "engineering_tests": ["tests/test_multidataset_pipeline.py::BEATsTemporalContractTest", "tests/test_hf_data.py::HFDataContractTest"],
    "read_only_verifier_entry": "python -m baseline.multidataset_pipeline.verify_hf_contract --phase all",
    "update_budget": None,
    "selection": None,
    "output_dir": "result/reproduce/P6_beats_shared_temporal",
    "receipt_path": "result/reproduce/P6_beats_shared_temporal/P6_receipt.json",
}
PROJECT_ROOT = Path(os.environ.get("ACOUSTIC_PROJECT_ROOT", Path.cwd())).resolve()
APPROVAL_RECEIPT = os.environ.get("P6_APPROVAL_RECEIPT")


## 科学 gate 与运行前审批

BEATs tokens [B,L,768] 必须先经过与 P2 同一个 shared Linear 768→256，再由 minimal Linear 256→4 输出四通道 logits；loss 为逐通道 masked BCE 后等权平均。native metrics 按 I/E/CAS/DAS 分通道报告 accuracy、ROC-AUC、AP、sensitivity、specificity、PPV、F1，threshold 只在 validation 选择。执行前仍需通过 exact token mask/time-map、HF semantics、ICBHI/SPRSound/KAUH shared-window route parity、真实 checkpoint CUDA smoke、approval 与新的 refinement verifier。任何 gate 未通过均 HOLD。

In [ ]:
PREFLIGHT = freeze_receipt()
TEMPORAL_HEAD = P6TokenTemporalHead  # class reference only; cell remains unexecuted
TEMPORAL_LOSS = hf_masked_channel_balanced_bce
required = [PIPELINE["update_budget"], PIPELINE["selection"], APPROVAL_RECEIPT]
if any(value in (None, "") for value in required):
    raise RuntimeError("P6 HOLD: temporal contract, parity, budget, selection, and approval must be verified")
approval_path = PROJECT_ROOT / APPROVAL_RECEIPT
if not approval_path.is_file():
    raise FileNotFoundError(approval_path)
DRY_RUN_PLAN = {"pipeline": PIPELINE, "contract": TEMPORAL_CONTRACT, "execute": False}


## 输出、receipt 与结果表

receipt 必须含 temporal schema version、device、token/mask/time-map geometry、I/E/CAS/DAS channel order、frozen head/loss、target policy、alignment=token_center_in_interval、negative_semantics=source_task_constructed_not_raw_normal、shared_label_eligible=false、逐通道 positive/constructed-negative/masked counts与denominator、validation-only thresholds、checkpoint/frontend hashes、window/crop lineage、padding verifier、non-HF parity、native/time metrics、seed/updates/selection 与 warnings。

| Gate / comparison | Result | Decision |
|---|---:|---|
| ICBHI/SPRSound/KAUH route parity；HF token package P6↔P2 | Not run | HOLD |

**Test Result = Not run。Decision = Deferred/HOLD。Claim boundary：P6−P2 是 BEATs token-level temporal refinement package effect；不是首轮 encoder shortlist 必需条件，也不是纯 encoder/head attribution。**